In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
    --------------------------------------- 1.6/69.5 MB 8.4 MB/s eta 0:00:09
   -- ------------------------------------- 3.9/69.5 MB 9.9 MB/s eta 0:00:07
   ---- ----------------------------------- 8.4/69.5 MB 13.8 MB/s eta 0:00:05
   -------- ------------------------------- 14.9/69.5 MB 18.3 MB/s eta 0:00:03
   ------------ --------------------------- 21.0/69.5 MB 20.5 MB/s eta 0:00:03
   ----------------- ---------------------- 29.6/69.5 MB 24.0 MB/s eta 0:00:02
   ---------------------- ----------------- 38.5/69.5 MB 26.9 MB/s eta 0:00:02
   --------------------------- ------------ 47.7/69.5 MB 29.2 MB/s eta 0:00:01
   --------------------------------- ------ 58.5/69.5 MB 31.6 MB/s eta 0:00:01
   ---------------------------------------  67.9/69.5 MB 33.0 MB/s eta 0:00:01
   ---------------------------------------  69.5/69.5 MB 33.2 MB/s eta 0:00


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
"""
01_prepare_data.py — Frozen Protocol v3 data preparation (PhD thesis pipeline).

Produces every artifact Scripts 02 (train XGB-A/B/C/D) and 03 (predict 142) need:
  artifacts/
    benchmark_split_v3.csv        molecules with >=1 of Tc/Pc/omega + split + cluster_id
    X_benchmark.npy               RDKit-2D + Morgan(2048,counts=False) features, benchmark order
    X_est.npy, est_pool.csv       leakage-filtered estimated pool (variant B/D) + features
    X_142.npy, refrigerants.csv   inference set + features
    tb_oof_benchmark.npy          out-of-fold predicted Tb for benchmark molecules (variant C/D)
    tb_pred_est.npy, tb_pred_142.npy   predicted Tb for estimated pool and 142 (same Tb model)
    tb_model.json                 final Tb model (trained on all Tb rows excl. val/test/142)
    feature_names.json, manifest.json  reproducibility record (hashes, dims, counts)

FAILS LOUDLY on: invalid SMILES, duplicate canonical structures within a file,
val/test/142 leakage into estimated pool or Tb training, feature-dimension mismatch.

Canonical identity everywhere: Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
Frozen split: Morgan r2/2048 -> Butina distance cutoff 0.60 -> deterministic
normalized-deficit assignment 80/10/10, computed on ALL benchmark molecules
BEFORE any target-specific label filtering.

Run: python 01_prepare_data.py
"""

# ================= CONFIG =================
EXPT_FILE     = "final_master_experimental_finetuning_6644.csv"
EST_FILE      = "final_pretraining_estimated_critprop.csv"
REFRIG_FILE   = "molecules_144_refrigerant.xlsx"
TARGETS       = ["Tc (K)", "Pc (bar)", "omega (-)"]
TB_COL        = "Tb (K)"
CLUSTER_CUTOFF, SPLIT_FP_BITS = 0.60, 2048
MODEL_FP_BITS = 2048
N_TB_OOF_FOLDS = 5            # cluster-aware GroupKFold for OOF Tb
SEED = 42
TB_XGB_PARAMS = dict(n_estimators=1500, learning_rate=0.03, max_depth=7, subsample=0.8,
                     colsample_bytree=0.7, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                     random_state=SEED, n_jobs=-1, tree_method="hist")
OUT = "artifacts"
# ==========================================

import os, json, hashlib, warnings
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor
RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")

DESC = [(n, f) for n, f in Descriptors.descList]
MGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MODEL_FP_BITS)
SGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=SPLIT_FP_BITS)


def canonical(s):
    m = Chem.MolFromSmiles(str(s)) if pd.notna(s) else None
    if m is None:
        raise ValueError(f"Invalid SMILES: {s!r}")
    return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)


def canonicalize_column(df, col, label):
    df = df.copy()
    df["canonical_smiles"] = df[col].map(canonical)
    dup = df["canonical_smiles"].duplicated()
    if dup.any():
        raise ValueError(f"Duplicate canonical structures in {label}: "
                         f"{df.loc[dup, 'canonical_smiles'].tolist()[:10]} ...")
    return df


def featurize(canonical_smiles):
    rows = []
    fdim = None
    for s in canonical_smiles:
        m = Chem.MolFromSmiles(s)
        d = []
        for _, fn in DESC:
            try:
                v = fn(m); d.append(v if np.isfinite(v) else 0.0)
            except Exception:
                d.append(0.0)
        row = np.concatenate([np.asarray(d, np.float32),
                              np.asarray(MGEN.GetFingerprint(m), np.float32)])
        if fdim is None:
            fdim = row.shape[0]
        elif row.shape[0] != fdim:
            raise ValueError(f"Feature-dimension mismatch for {s}: {row.shape[0]} != {fdim}")
        rows.append(row)
    return np.vstack(rows)


def frozen_cluster_split(canonical_smiles, cutoff=CLUSTER_CUTOFF, fp_bits=SPLIT_FP_BITS,
                         fractions=(0.8, 0.1, 0.1)):
    fps = [SGEN.GetFingerprint(Chem.MolFromSmiles(s)) for s in canonical_smiles]
    n = len(fps); dists = []
    for i in range(1, n):
        dists.extend(1 - x for x in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i]))
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    names = ("train", "val", "test")
    target = {k: f * n for k, f in zip(names, fractions)}
    cnt = {k: 0 for k in names}
    split = np.empty(n, object); cid = np.zeros(n, int)
    for c, members in sorted(enumerate(clusters), key=lambda z: (-len(z[1]), z[0])):
        deficit = {k: (target[k] - cnt[k]) / target[k] for k in names}
        q = max(names, key=lambda k: deficit[k])
        for i in members:
            split[i] = q; cid[i] = c
        cnt[q] += len(members)
    return split, cid


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def main():
    os.makedirs(OUT, exist_ok=True)

    # ---------- load & canonicalize (fail loudly) ----------
    expt = canonicalize_column(pd.read_csv(EXPT_FILE), "smiles", "experimental_file")
    est = canonicalize_column(pd.read_csv(EST_FILE), "smiles", "estimated_file")
    refrig = canonicalize_column(pd.read_excel(REFRIG_FILE), "SMILES", "refrigerant_file")
    refrig_set = set(refrig.canonical_smiles)

    # refrigerants must be absent from BOTH training files (canonical identity)
    leak_e = set(expt.canonical_smiles) & refrig_set
    leak_p = set(est.canonical_smiles) & refrig_set
    if leak_e or leak_p:
        raise ValueError(f"Refrigerant leakage! expt: {sorted(leak_e)[:5]} est: {sorted(leak_p)[:5]}")

    # ---------- benchmark set: >=1 of Tc/Pc/omega ----------
    bench = expt[expt[TARGETS].notna().any(axis=1)].reset_index(drop=True)
    label_counts = {t: int(bench[t].notna().sum()) for t in TARGETS}
    print(f"Benchmark molecules: {len(bench)} | labels: {label_counts}")

    # ---------- frozen split v3 (before label filtering) ----------
    split, cid = frozen_cluster_split(bench.canonical_smiles.tolist())
    bench["split"], bench["cluster_id"] = split, cid
    sizes = pd.Series(split).value_counts().to_dict()
    print("Split v3 sizes:", sizes)
    # sanity: no cluster in two partitions
    if bench.groupby("cluster_id")["split"].nunique().max() != 1:
        raise ValueError("Cluster leakage across partitions!")
    bench.to_csv(f"{OUT}/benchmark_split_v3.csv", index=False)

    banned = set(bench.loc[bench.split.isin(["val", "test"]), "canonical_smiles"]) | refrig_set

    # ---------- estimated pool for variants B/D (leakage-filtered) ----------
    est_pool = est[~est.canonical_smiles.isin(banned)].reset_index(drop=True)
    removed = len(est) - len(est_pool)
    print(f"Estimated pool: {len(est_pool)} (removed {removed} val/test/142-overlapping molecules)")
    if set(est_pool.canonical_smiles) & banned:
        raise ValueError("Estimated pool still contains banned molecules!")
    est_pool.to_csv(f"{OUT}/est_pool.csv", index=False)

    # ---------- features (identical featurizer everywhere) ----------
    X_bench = featurize(bench.canonical_smiles); np.save(f"{OUT}/X_benchmark.npy", X_bench)
    X_est = featurize(est_pool.canonical_smiles); np.save(f"{OUT}/X_est.npy", X_est)
    X_142 = featurize(refrig.canonical_smiles); np.save(f"{OUT}/X_142.npy", X_142)
    if not (X_bench.shape[1] == X_est.shape[1] == X_142.shape[1]):
        raise ValueError("Feature-dimension mismatch across matrices!")
    refrig.to_csv(f"{OUT}/refrigerants.csv", index=False)
    feat_names = [n for n, _ in DESC] + [f"morgan_{i}" for i in range(MODEL_FP_BITS)]
    json.dump(feat_names, open(f"{OUT}/feature_names.json", "w"))

    # ---------- Tb model + OOF predicted Tb (variants C/D) ----------
    # Tb training pool: all experimental rows with Tb whose canonical SMILES is NOT
    # in benchmark val/test and NOT a refrigerant. Benchmark-train molecules allowed.
    tb_rows = expt[expt[TB_COL].notna() & ~expt.canonical_smiles.isin(banned)].reset_index(drop=True)
    print(f"Tb training rows (leakage-filtered): {len(tb_rows)}")
    X_tb = featurize(tb_rows.canonical_smiles)
    y_tb = tb_rows[TB_COL].values.astype(float)

    # cluster ids for Tb pool: reuse benchmark clusters where molecule is in benchmark,
    # else give each remaining molecule its own group via a fresh Butina on the pool.
    # Simpler + still leakage-safe for OOF purposes: Butina on the Tb pool itself.
    tb_split, tb_cid = frozen_cluster_split(tb_rows.canonical_smiles.tolist())  # only cid used
    oof_tb_pool = np.zeros(len(tb_rows))
    gkf = GroupKFold(n_splits=N_TB_OOF_FOLDS)
    for a, b in gkf.split(X_tb, groups=tb_cid):
        m = XGBRegressor(**TB_XGB_PARAMS); m.fit(X_tb[a], y_tb[a])
        oof_tb_pool[b] = m.predict(X_tb[b])
    tb_rmse = float(np.sqrt(np.mean((oof_tb_pool - y_tb) ** 2)))
    print(f"Tb OOF RMSE (cluster-aware, pool): {tb_rmse:.2f} K")

    # final Tb model on the full (filtered) pool -> used for val/test/est/142 Tb features
    tb_model = XGBRegressor(**TB_XGB_PARAMS); tb_model.fit(X_tb, y_tb)
    tb_model.save_model(f"{OUT}/tb_model.json")

    # predicted-Tb feature for every benchmark molecule:
    #   train-partition molecules that are in the Tb pool -> their OOF prediction (no self-leak)
    #   all other benchmark molecules (incl. val/test)    -> final-model prediction
    pool_oof = dict(zip(tb_rows.canonical_smiles, oof_tb_pool))
    tb_feat_bench = np.empty(len(bench))
    final_pred_bench = tb_model.predict(X_bench)
    for i, (cs, sp) in enumerate(zip(bench.canonical_smiles, bench.split)):
        if sp == "train" and cs in pool_oof:
            tb_feat_bench[i] = pool_oof[cs]
        else:
            tb_feat_bench[i] = final_pred_bench[i]
    np.save(f"{OUT}/tb_oof_benchmark.npy", tb_feat_bench)
    np.save(f"{OUT}/tb_pred_est.npy", tb_model.predict(X_est))
    np.save(f"{OUT}/tb_pred_142.npy", tb_model.predict(X_142))

    # ---------- manifest ----------
    manifest = {
        "protocol": "frozen_v3",
        "canonical_definition": "Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)",
        "split": {"cutoff": CLUSTER_CUTOFF, "fp_bits": SPLIT_FP_BITS,
                  "assignment": "deterministic normalized-deficit 80/10/10", "sizes": sizes},
        "benchmark_n": len(bench),
        "labels": {t: int(bench[t].notna().sum()) for t in TARGETS},
        "est_pool_n": len(est_pool), "est_removed_for_leakage": removed,
        "tb_pool_n": len(tb_rows), "tb_oof_rmse_K": tb_rmse,
        "feature_dim": int(X_bench.shape[1]),
        "input_hashes": {EXPT_FILE: sha256(EXPT_FILE), EST_FILE: sha256(EST_FILE),
                          REFRIG_FILE: sha256(REFRIG_FILE)},
        "seed": SEED,
    }
    json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=2)
    print(json.dumps(manifest, indent=2))
    print(f"\nAll artifacts saved to {OUT}/ — Script 02 consumes these directly.")


if __name__ == "__main__":
    main()

Benchmark molecules: 2087 | labels: {'Tc (K)': 2086, 'Pc (bar)': 1950, 'omega (-)': 1513}
Split v3 sizes: {'train': 1669, 'val': 209, 'test': 209}
Estimated pool: 5803 (removed 338 val/test/142-overlapping molecules)
Tb training rows (leakage-filtered): 5318
Tb OOF RMSE (cluster-aware, pool): 25.65 K
{
  "protocol": "frozen_v3",
  "canonical_definition": "Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)",
  "split": {
    "cutoff": 0.6,
    "fp_bits": 2048,
    "assignment": "deterministic normalized-deficit 80/10/10",
    "sizes": {
      "train": 1669,
      "val": 209,
      "test": 209
    }
  },
  "benchmark_n": 2087,
  "labels": {
    "Tc (K)": 2086,
    "Pc (bar)": 1950,
    "omega (-)": 1513
  },
  "est_pool_n": 5803,
  "est_removed_for_leakage": 338,
  "tb_pool_n": 5318,
  "tb_oof_rmse_K": 25.645574645885688,
  "feature_dim": 2265,
  "input_hashes": {
    "final_master_experimental_finetuning_6644.csv": "a2c259fd40a8a6bea98a4e3794d365cb56dc55c5c5b7a744bcccb055fb9967e

In [7]:
"""
02_train_variants.py — XGB-A/B/C/D on Frozen Protocol v3. Consumes artifacts/ from Script 01.

| Variant | Design                                                       |
| A       | Morgan + RDKit, experimental only                            |
| B       | A + low-weight estimated labels (sample_weight = W_EST)      |
| C       | A + OOF-predicted Tb feature                                 |
| D       | C + low-weight estimated labels                              |

Rules enforced:
  - identical frozen split v3 for all variants (loaded, never recomputed)
  - val used ONLY for early stopping; test evaluated once per (variant,target)
  - estimated rows never appear in val or test (guaranteed by Script 01; re-checked)

Run: python 02_train_variants.py
"""
# ================= CONFIG =================
ART = "artifacts"
TARGETS = ["Tc (K)", "Pc (bar)", "omega (-)"]
W_EST = 0.2                      # sample weight for estimated labels (variants B, D)
XGB_PARAMS = dict(n_estimators=3000, learning_rate=0.03, max_depth=7, subsample=0.8,
                  colsample_bytree=0.7, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                  random_state=42, n_jobs=-1, tree_method="hist",
                  early_stopping_rounds=150)
OUT = "results_v3"
# ==========================================
import os, json
import numpy as np, pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def metric_row(y, p):
    e = np.abs(y - p)
    return dict(n=len(y), R2=r2_score(y, p), RMSE=np.sqrt(mean_squared_error(y, p)),
                MAE=mean_absolute_error(y, p), MedAE=np.median(e), MaxAE=e.max())

def main():
    os.makedirs(OUT, exist_ok=True)
    bench = pd.read_csv(f"{ART}/benchmark_split_v3.csv")
    est = pd.read_csv(f"{ART}/est_pool.csv")
    X = np.load(f"{ART}/X_benchmark.npy")
    Xe = np.load(f"{ART}/X_est.npy")
    tb_bench = np.load(f"{ART}/tb_oof_benchmark.npy")
    tb_est = np.load(f"{ART}/tb_pred_est.npy")

    # re-check leakage invariants (fail loudly)
    banned = set(bench.loc[bench.split.isin(["val", "test"]), "canonical_smiles"])
    if set(est.canonical_smiles) & banned:
        raise ValueError("Estimated pool intersects val/test — artifacts corrupted.")

    tr = bench.split.values == "train"
    va = bench.split.values == "val"
    te = bench.split.values == "test"

    # feature matrices per variant
    X_plain = X
    X_tb = np.hstack([X, tb_bench[:, None]])
    Xe_plain = Xe
    Xe_tb = np.hstack([Xe, tb_est[:, None]])

    VARIANTS = {
        "XGB-A": dict(Xb=X_plain, Xest=None),
        "XGB-B": dict(Xb=X_plain, Xest=Xe_plain),
        "XGB-C": dict(Xb=X_tb,    Xest=None),
        "XGB-D": dict(Xb=X_tb,    Xest=Xe_tb),
    }

    rows = []
    test_pred_frames = []
    for vname, cfg in VARIANTS.items():
        for tgt in TARGETS:
            has = bench[tgt].notna().values
            m_tr, m_va, m_te = has & tr, has & va, has & te
            Xtr, ytr = cfg["Xb"][m_tr], bench[tgt].values[m_tr].astype(float)
            wtr = np.ones(len(ytr))
            if cfg["Xest"] is not None:
                ehas = est[tgt].notna().values
                Xtr = np.vstack([Xtr, cfg["Xest"][ehas]])
                ytr = np.concatenate([ytr, est[tgt].values[ehas].astype(float)])
                wtr = np.concatenate([wtr, np.full(int(ehas.sum()), W_EST)])
            model = XGBRegressor(**XGB_PARAMS)
            model.fit(Xtr, ytr, sample_weight=wtr,
                      eval_set=[(cfg["Xb"][m_va], bench[tgt].values[m_va].astype(float))],
                      verbose=False)
            model.save_model(f"{OUT}/{vname}_{tgt.split(' ')[0]}.json")
            yte = bench[tgt].values[m_te].astype(float)
            pte = model.predict(cfg["Xb"][m_te])
            rows.append({"variant": vname, "target": tgt,
                         **{k: round(float(v), 4) for k, v in metric_row(yte, pte).items()},
                         "best_iteration": int(model.best_iteration),
                         "n_train_expt": int(m_tr.sum()),
                         "n_train_est": 0 if cfg["Xest"] is None else int(est[tgt].notna().sum())})
            f = bench.loc[m_te, ["canonical_smiles", "cluster_id"]].copy()
            f["variant"], f["target"] = vname, tgt
            f["actual"], f["pred"] = yte, pte
            test_pred_frames.append(f)

    res = pd.DataFrame(rows)
    res.to_csv(f"{OUT}/variant_test_metrics.csv", index=False)
    pd.concat(test_pred_frames).to_csv(f"{OUT}/variant_test_predictions.csv", index=False)
    print(res.to_string(index=False))
    print(f"\nSaved to {OUT}/")

if __name__ == "__main__":
    main()

variant    target     n     R2    RMSE     MAE   MedAE    MaxAE  best_iteration  n_train_expt  n_train_est
  XGB-A    Tc (K) 209.0 0.9235 31.1722 19.8383 11.0042 133.0551             789          1668            0
  XGB-A  Pc (bar) 195.0 0.8816  3.5689  1.8778  0.9024  21.8438             467          1559            0
  XGB-A omega (-) 157.0 0.9197  0.0829  0.0495  0.0199   0.3590             337          1221            0
  XGB-B    Tc (K) 209.0 0.9383 27.9880 18.4898 11.3593 144.9265             190          1668         5554
  XGB-B  Pc (bar) 195.0 0.8754  3.6614  1.8777  0.8601  26.8268             102          1559         5616
  XGB-B omega (-) 157.0 0.9137  0.0860  0.0460  0.0142   0.3736             498          1221         4673
  XGB-C    Tc (K) 209.0 0.9457 26.2513 17.0181 10.9718 123.7222            1052          1668            0
  XGB-C  Pc (bar) 195.0 0.8782  3.6197  1.8901  0.8806  23.8456             515          1559            0
  XGB-C omega (-) 157.0 0.9226  0.081

In [8]:
"""
03a_analysis_plots.py — Analysis plots & matrices for XGB-A/B/C/D on Frozen Protocol v3.
Consumes: artifacts/benchmark_split_v3.csv, results_v3/variant_test_predictions.csv,
          results_v3/variant_test_metrics.csv
Produces (in results_v3/plots/):
  parity_<variant>.png            3-panel parity per variant, colored by max-Tanimoto
  residuals_<variant>.png         residual-vs-predicted per target
  variant_comparison.png          grouped bar: RMSE per target per variant
  error_vs_similarity.png         |error| vs max-Tanimoto for best variant (XGB-D)
  similarity_binned_mae.csv       MAE per Tanimoto bin per variant per target
  worst20_<target>.csv            20 largest test errors (best variant) with similarity
  metrics_matrix.csv              pivoted variant x target metric matrix

Run: python 03a_analysis_plots.py
"""
ART, RES = "artifacts", "results_v3"
BEST_VARIANT = "XGB-D"
BINS = [0, 0.4, 0.6, 0.8, 1.01]; BIN_LABELS = ["<0.4", "0.4-0.6", "0.6-0.8", ">0.8"]

import os
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
RDLogger.DisableLog("rdApp.*")

def main():
    outdir = f"{RES}/plots"; os.makedirs(outdir, exist_ok=True)
    bench = pd.read_csv(f"{ART}/benchmark_split_v3.csv")
    preds = pd.read_csv(f"{RES}/variant_test_predictions.csv")
    mets = pd.read_csv(f"{RES}/variant_test_metrics.csv")

    # ---- max Tanimoto (test -> train), 2048-bit protocol fingerprints ----
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = {cs: gen.GetFingerprint(Chem.MolFromSmiles(cs)) for cs in bench.canonical_smiles}
    ftr = [fps[cs] for cs in bench.loc[bench.split == "train", "canonical_smiles"]]
    te_cs = bench.loc[bench.split == "test", "canonical_smiles"]
    maxsim = {cs: max(DataStructs.BulkTanimotoSimilarity(fps[cs], ftr)) for cs in te_cs}
    preds["max_tanimoto"] = preds.canonical_smiles.map(maxsim)
    preds["abs_error"] = (preds.actual - preds.pred).abs()
    preds["sim_bin"] = pd.cut(preds.max_tanimoto, BINS, labels=BIN_LABELS)

    targets = preds.target.unique().tolist()
    variants = preds.variant.unique().tolist()

    # ---- parity + residual plots per variant ----
    for v in variants:
        fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4.5))
        for ax, t in zip(np.atleast_1d(axes), targets):
            d = preds[(preds.variant == v) & (preds.target == t)]
            sc = ax.scatter(d.actual, d.pred, s=12, alpha=0.6, c=d.max_tanimoto,
                            cmap="viridis", vmin=0, vmax=1)
            lo, hi = min(d.actual.min(), d.pred.min()), max(d.actual.max(), d.pred.max())
            ax.plot([lo, hi], [lo, hi], "k-", lw=1)
            r2 = 1 - ((d.actual-d.pred)**2).sum()/((d.actual-d.actual.mean())**2).sum()
            ax.set_title(f"{v}  {t}  (R2={r2:.3f})"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        fig.colorbar(sc, ax=axes, label="max Tanimoto to train", shrink=0.8)
        plt.savefig(f"{outdir}/parity_{v}.png", dpi=150, bbox_inches="tight"); plt.close()

        fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
        for ax, t in zip(np.atleast_1d(axes), targets):
            d = preds[(preds.variant == v) & (preds.target == t)]
            ax.scatter(d.pred, d.actual - d.pred, s=12, alpha=0.6)
            ax.axhline(0, color="k", lw=1)
            ax.set_title(f"{v}  {t}"); ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
        plt.tight_layout(); plt.savefig(f"{outdir}/residuals_{v}.png", dpi=150); plt.close()

    # ---- variant comparison bars (RMSE per target) ----
    fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
    for ax, t in zip(np.atleast_1d(axes), targets):
        d = mets[mets.target == t]
        ax.bar(d.variant, d.RMSE, color="tab:blue", alpha=0.8)
        ax.set_title(f"{t} — test RMSE"); ax.tick_params(axis="x", rotation=20)
        for x, y in zip(d.variant, d.RMSE): ax.text(x, y, f"{y:.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{outdir}/variant_comparison.png", dpi=150); plt.close()

    # ---- error vs similarity (best variant) ----
    fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
    for ax, t in zip(np.atleast_1d(axes), targets):
        d = preds[(preds.variant == BEST_VARIANT) & (preds.target == t)]
        ax.scatter(d.max_tanimoto, d.abs_error, s=12, alpha=0.6)
        r = np.corrcoef(d.max_tanimoto, d.abs_error)[0, 1]
        ax.set_title(f"{BEST_VARIANT}  {t}  (r={r:.2f})")
        ax.set_xlabel("max Tanimoto to train"); ax.set_ylabel("|error|")
    plt.tight_layout(); plt.savefig(f"{outdir}/error_vs_similarity.png", dpi=150); plt.close()

    # ---- similarity-binned MAE matrix ----
    binned = (preds.groupby(["variant", "target", "sim_bin"])
              .agg(n=("abs_error", "size"), MAE=("abs_error", "mean")).round(4).reset_index())
    binned.to_csv(f"{outdir}/similarity_binned_mae.csv", index=False)

    # ---- worst-20 per target (best variant) ----
    for t in targets:
        d = preds[(preds.variant == BEST_VARIANT) & (preds.target == t)]
        d.nlargest(20, "abs_error").to_csv(
            f"{outdir}/worst20_{t.split(' ')[0]}.csv", index=False)

    # ---- pivoted metric matrix ----
    mets.pivot_table(index="variant", columns="target",
                     values=["R2", "RMSE", "MAE", "MedAE", "MaxAE"]).round(4)\
        .to_csv(f"{outdir}/metrics_matrix.csv")

    print(binned[binned.variant == BEST_VARIANT].to_string(index=False))
    print(f"\nAll plots and matrices saved to {outdir}/")

if __name__ == "__main__":
    main()

variant    target sim_bin   n     MAE
  XGB-D  Pc (bar)    <0.4  34  3.6194
  XGB-D  Pc (bar) 0.4-0.6 105  1.6245
  XGB-D  Pc (bar) 0.6-0.8  50  1.2536
  XGB-D  Pc (bar)    >0.8   6  1.0753
  XGB-D    Tc (K)    <0.4  38 23.0681
  XGB-D    Tc (K) 0.4-0.6 111 16.1020
  XGB-D    Tc (K) 0.6-0.8  54 12.0753
  XGB-D    Tc (K)    >0.8   6 14.7908
  XGB-D omega (-)    <0.4  27  0.0500
  XGB-D omega (-) 0.4-0.6  83  0.0356
  XGB-D omega (-) 0.6-0.8  42  0.0596
  XGB-D omega (-)    >0.8   5  0.0824

All plots and matrices saved to results_v3/plots/


In [9]:
"""
03_predict_refrigerants.py — Final Tc/Pc/omega predictions for the 142 refrigerant
candidates using XGB-D (Morgan + RDKit + predicted Tb + low-weight estimated labels),
the best variant from Script 02 on Frozen Protocol v3.

Consumes artifacts/ (Script 01) and est_pool/tb artifacts. Retrains XGB-D deployment
models on ALL available labeled data (train+val+test) per target — standard once the
frozen benchmark evaluation is complete and locked.

Uncertainty per prediction (two independent signals, both reported):
  1. ensemble_std   — std across N_SEEDS seed-perturbed XGB-D models (epistemic)
  2. max_tanimoto   — nearest-neighbour similarity to the experimental TRAIN molecules
  3. trust_flag     — HIGH (sim>=0.6), MEDIUM (0.4-0.6), LOW (<0.4);
                      LOW predictions are extrapolations — flag in thesis, don't over-trust.

Output: results_v3/refrigerant_142_final_predictions.csv
        results_v3/plots/refrigerant_trust_summary.png

Run: python 03_predict_refrigerants.py
"""
# ================= CONFIG =================
ART, RES = "artifacts", "results_v3"
TARGETS = ["Tc (K)", "Pc (bar)", "omega (-)"]
W_EST = 0.2
N_SEEDS = 25                      # ensemble for epistemic uncertainty (Biswas-style)
XGB_PARAMS = dict(n_estimators=800, learning_rate=0.03, max_depth=7, subsample=0.8,
                  colsample_bytree=0.7, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                  n_jobs=-1, tree_method="hist")   # fixed n_estimators (no early stop at deploy)
TRUST = [(0.6, "HIGH"), (0.4, "MEDIUM"), (0.0, "LOW")]
# ==========================================
import os, json
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from xgboost import XGBRegressor
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
RDLogger.DisableLog("rdApp.*")

def trust_flag(sim):
    for thr, lab in TRUST:
        if sim >= thr: return lab
    return "LOW"

def main():
    os.makedirs(f"{RES}/plots", exist_ok=True)
    bench = pd.read_csv(f"{ART}/benchmark_split_v3.csv")
    est = pd.read_csv(f"{ART}/est_pool.csv")
    refrig = pd.read_csv(f"{ART}/refrigerants.csv")
    X = np.load(f"{ART}/X_benchmark.npy"); Xe = np.load(f"{ART}/X_est.npy"); X142 = np.load(f"{ART}/X_142.npy")
    tb_bench = np.load(f"{ART}/tb_oof_benchmark.npy")
    tb_est = np.load(f"{ART}/tb_pred_est.npy"); tb_142 = np.load(f"{ART}/tb_pred_142.npy")

    # XGB-D feature stack: [features | predicted Tb]
    Xb = np.hstack([X, tb_bench[:, None]])
    Xed = np.hstack([Xe, tb_est[:, None]])
    X142d = np.hstack([X142, tb_142[:, None]])

    # similarity of each refrigerant to experimental TRAIN molecules
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    ftr = [gen.GetFingerprint(Chem.MolFromSmiles(s))
           for s in bench.loc[bench.split == "train", "canonical_smiles"]]
    f142 = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in refrig.canonical_smiles]
    maxsim = np.array([max(DataStructs.BulkTanimotoSimilarity(f, ftr)) for f in f142])

    out = refrig[["IUPAC_Name", "SMILES", "canonical_smiles"]].copy() \
        if "IUPAC_Name" in refrig.columns else refrig[["SMILES", "canonical_smiles"]].copy()
    out["max_tanimoto_to_train"] = maxsim.round(3)
    out["trust_flag"] = [trust_flag(s) for s in maxsim]
    out["exact_training_overlap"] = out.canonical_smiles.isin(set(bench.canonical_smiles))  # expect all False

    for tgt in TARGETS:
        has = bench[tgt].notna().values
        Xtr = np.vstack([Xb[has], Xed[est[tgt].notna().values]])
        ytr = np.concatenate([bench[tgt].values[has].astype(float),
                              est[tgt].values[est[tgt].notna().values].astype(float)])
        wtr = np.concatenate([np.ones(int(has.sum())),
                              np.full(int(est[tgt].notna().sum()), W_EST)])
        preds = []
        for seed in range(N_SEEDS):
            m = XGBRegressor(**XGB_PARAMS, random_state=seed)
            m.fit(Xtr, ytr, sample_weight=wtr)
            preds.append(m.predict(X142d))
        preds = np.vstack(preds)
        short = tgt.split(" ")[0]
        out[f"{short}_pred"] = preds.mean(0).round(4)
        out[f"{short}_ensemble_std"] = preds.std(0).round(4)

    out.to_csv(f"{RES}/refrigerant_142_final_predictions.csv", index=False)

    # trust summary plot
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    vc = out.trust_flag.value_counts().reindex(["HIGH", "MEDIUM", "LOW"]).fillna(0)
    ax[0].bar(vc.index, vc.values, color=["tab:green", "tab:orange", "tab:red"])
    ax[0].set_title("Refrigerant predictions by trust flag"); ax[0].set_ylabel("count")
    for x, y in zip(vc.index, vc.values): ax[0].text(x, y, int(y), ha="center", va="bottom")
    ax[1].hist(maxsim, bins=20, color="tab:blue", alpha=0.8)
    ax[1].axvline(0.4, color="r", ls="--"); ax[1].axvline(0.6, color="orange", ls="--")
    ax[1].set_title("Max Tanimoto of 142 to training set"); ax[1].set_xlabel("max Tanimoto")
    plt.tight_layout(); plt.savefig(f"{RES}/plots/refrigerant_trust_summary.png", dpi=150)

    print(out[["SMILES", "trust_flag"] + [f"{t.split(' ')[0]}_pred" for t in TARGETS]].head(15).to_string(index=False))
    print("\nTrust distribution:", out.trust_flag.value_counts().to_dict())
    print(f"Exact training overlaps (should be 0): {int(out.exact_training_overlap.sum())}")
    print(f"Saved: {RES}/refrigerant_142_final_predictions.csv")

if __name__ == "__main__":
    main()

                 SMILES trust_flag    Tc_pred   Pc_pred  omega_pred
                     CC        LOW 298.016510 46.915901      0.1080
                    C#C        LOW 296.791412 50.004299      0.1043
                   C=CC        LOW 412.873199 46.721901      0.1659
                    CCC     MEDIUM 399.262299 43.734600      0.1762
                   C#CC        LOW 426.694397 53.265598      0.1484
                  C1CC1       HIGH 401.026398 58.405102      0.1447
                 CC(C)C     MEDIUM 427.795197 38.783699      0.1936
                 C1=CC1     MEDIUM 410.376404 57.878101      0.1404
                     CF        LOW 419.894012 74.266998      0.2897
                    FCF     MEDIUM 394.352997 58.089901      0.2497
                    CCF        LOW 436.681000 50.290901      0.2401
                 CC(F)F       HIGH 409.018188 45.928101      0.2425
             FC(F)C(F)F     MEDIUM 401.697998 41.114300      0.2630
FC(F)C1(F)C(F)(F)C1(F)F        LOW 423.228607 31

In [10]:
"""
03a_analysis_plots.py — Analysis plots & matrices for XGB-A/B/C/D on Frozen Protocol v3.
Consumes: artifacts/benchmark_split_v3.csv, results_v3/variant_test_predictions.csv,
          results_v3/variant_test_metrics.csv
Produces (in results_v3/plots/):
  parity_<variant>.png            3-panel parity per variant, colored by max-Tanimoto
  residuals_<variant>.png         residual-vs-predicted per target
  variant_comparison.png          grouped bar: RMSE per target per variant
  error_vs_similarity.png         |error| vs max-Tanimoto for best variant (XGB-D)
  similarity_binned_mae.csv       MAE per Tanimoto bin per variant per target
  worst20_<target>.csv            20 largest test errors (best variant) with similarity
  metrics_matrix.csv              pivoted variant x target metric matrix

Run: python 03a_analysis_plots.py
"""
ART, RES = "artifacts", "results_v3"
BEST_VARIANT = "XGB-D"
BINS = [0, 0.4, 0.6, 0.8, 1.01]; BIN_LABELS = ["<0.4", "0.4-0.6", "0.6-0.8", ">0.8"]

import os
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
RDLogger.DisableLog("rdApp.*")

def main():
    outdir = f"{RES}/plots"; os.makedirs(outdir, exist_ok=True)
    bench = pd.read_csv(f"{ART}/benchmark_split_v3.csv")
    preds = pd.read_csv(f"{RES}/variant_test_predictions.csv")
    mets = pd.read_csv(f"{RES}/variant_test_metrics.csv")

    # ---- max Tanimoto (test -> train), 2048-bit protocol fingerprints ----
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = {cs: gen.GetFingerprint(Chem.MolFromSmiles(cs)) for cs in bench.canonical_smiles}
    ftr = [fps[cs] for cs in bench.loc[bench.split == "train", "canonical_smiles"]]
    te_cs = bench.loc[bench.split == "test", "canonical_smiles"]
    maxsim = {cs: max(DataStructs.BulkTanimotoSimilarity(fps[cs], ftr)) for cs in te_cs}
    preds["max_tanimoto"] = preds.canonical_smiles.map(maxsim)
    preds["abs_error"] = (preds.actual - preds.pred).abs()
    preds["sim_bin"] = pd.cut(preds.max_tanimoto, BINS, labels=BIN_LABELS)

    targets = preds.target.unique().tolist()
    variants = preds.variant.unique().tolist()

    # ---- parity + residual plots per variant ----
    for v in variants:
        fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4.5))
        for ax, t in zip(np.atleast_1d(axes), targets):
            d = preds[(preds.variant == v) & (preds.target == t)]
            sc = ax.scatter(d.actual, d.pred, s=12, alpha=0.6, c=d.max_tanimoto,
                            cmap="viridis", vmin=0, vmax=1)
            lo, hi = min(d.actual.min(), d.pred.min()), max(d.actual.max(), d.pred.max())
            ax.plot([lo, hi], [lo, hi], "k-", lw=1)
            r2 = 1 - ((d.actual-d.pred)**2).sum()/((d.actual-d.actual.mean())**2).sum()
            ax.set_title(f"{v}  {t}  (R2={r2:.3f})"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        fig.colorbar(sc, ax=axes, label="max Tanimoto to train", shrink=0.8)
        plt.savefig(f"{outdir}/parity_{v}.png", dpi=150, bbox_inches="tight"); plt.close()

        fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
        for ax, t in zip(np.atleast_1d(axes), targets):
            d = preds[(preds.variant == v) & (preds.target == t)]
            ax.scatter(d.pred, d.actual - d.pred, s=12, alpha=0.6)
            ax.axhline(0, color="k", lw=1)
            ax.set_title(f"{v}  {t}"); ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
        plt.tight_layout(); plt.savefig(f"{outdir}/residuals_{v}.png", dpi=150); plt.close()

    # ---- variant comparison bars (RMSE per target) ----
    fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
    for ax, t in zip(np.atleast_1d(axes), targets):
        d = mets[mets.target == t]
        ax.bar(d.variant, d.RMSE, color="tab:blue", alpha=0.8)
        ax.set_title(f"{t} — test RMSE"); ax.tick_params(axis="x", rotation=20)
        for x, y in zip(d.variant, d.RMSE): ax.text(x, y, f"{y:.2f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{outdir}/variant_comparison.png", dpi=150); plt.close()

    # ---- error vs similarity (best variant) ----
    fig, axes = plt.subplots(1, len(targets), figsize=(5*len(targets), 4))
    for ax, t in zip(np.atleast_1d(axes), targets):
        d = preds[(preds.variant == BEST_VARIANT) & (preds.target == t)]
        ax.scatter(d.max_tanimoto, d.abs_error, s=12, alpha=0.6)
        r = np.corrcoef(d.max_tanimoto, d.abs_error)[0, 1]
        ax.set_title(f"{BEST_VARIANT}  {t}  (r={r:.2f})")
        ax.set_xlabel("max Tanimoto to train"); ax.set_ylabel("|error|")
    plt.tight_layout(); plt.savefig(f"{outdir}/error_vs_similarity.png", dpi=150); plt.close()

    # ---- similarity-binned MAE matrix ----
    binned = (preds.groupby(["variant", "target", "sim_bin"])
              .agg(n=("abs_error", "size"), MAE=("abs_error", "mean")).round(4).reset_index())
    binned.to_csv(f"{outdir}/similarity_binned_mae.csv", index=False)

    # ---- worst-20 per target (best variant) ----
    for t in targets:
        d = preds[(preds.variant == BEST_VARIANT) & (preds.target == t)]
        d.nlargest(20, "abs_error").to_csv(
            f"{outdir}/worst20_{t.split(' ')[0]}.csv", index=False)

    # ---- pivoted metric matrix ----
    mets.pivot_table(index="variant", columns="target",
                     values=["R2", "RMSE", "MAE", "MedAE", "MaxAE"]).round(4)\
        .to_csv(f"{outdir}/metrics_matrix.csv")

    print(binned[binned.variant == BEST_VARIANT].to_string(index=False))
    print(f"\nAll plots and matrices saved to {outdir}/")

if __name__ == "__main__":
    main()

variant    target sim_bin   n     MAE
  XGB-D  Pc (bar)    <0.4  34  3.6194
  XGB-D  Pc (bar) 0.4-0.6 105  1.6245
  XGB-D  Pc (bar) 0.6-0.8  50  1.2536
  XGB-D  Pc (bar)    >0.8   6  1.0753
  XGB-D    Tc (K)    <0.4  38 23.0681
  XGB-D    Tc (K) 0.4-0.6 111 16.1020
  XGB-D    Tc (K) 0.6-0.8  54 12.0753
  XGB-D    Tc (K)    >0.8   6 14.7908
  XGB-D omega (-)    <0.4  27  0.0500
  XGB-D omega (-) 0.4-0.6  83  0.0356
  XGB-D omega (-) 0.6-0.8  42  0.0596
  XGB-D omega (-)    >0.8   5  0.0824

All plots and matrices saved to results_v3/plots/


In [12]:
"""
03_analyze_variants.py — Analysis-only research archive for the XGB-A/B/C/D ablation
on Frozen Protocol v3. Consumes SAVED artifacts and prediction CSVs only.
NO retraining, resplitting, retuning, or regeneration of predictions.

Ablation semantics (base features = Morgan 2048 + RDKit 2D descriptors):
  XGB-A = experimental + base
  XGB-B = experimental + weighted estimated + base          (A->B isolates estimated supervision)
  XGB-C = experimental + base + leakage-safe predicted Tb   (A->C isolates predicted Tb)
  XGB-D = experimental + weighted estimated + base + Tb      (C->D: est. given Tb; B->D: Tb given est.)

Single frozen split: NO error bars, NO statistical-significance claims. Every A->B / A->C /
C->D / B->D delta is "observed on this frozen test split" only.

Residual sign convention (stated everywhere): residual = actual - predicted.
  positive residual  => model UNDER-predicts.

Improvement convention for ablation deltas (lower error = better):
  delta_metric = metric(baseline) - metric(variant);  POSITIVE delta => variant improves (reduces error).

Negligibility rule (printed in summary):
  For each contrast+target, pool |errors| of the two variants, resolution = 0.5 * median(pooled |error|).
  |delta_RMSE| or |delta_MAE| below resolution => classified NEGLIGIBLE on this split.

Inputs:
  artifacts/benchmark_split_v3.csv
  results_v3/variant_test_metrics.csv
  results_v3/variant_test_predictions.csv   (variant,target,canonical_smiles,cluster_id,actual,pred)

Outputs: results_v3/analysis_v2/ (see tree at bottom of chat message)

Run: python 03_analyze_variants.py
"""
# ================= CONFIG =================
ART = "artifacts"
RES = "results_v3"
OUTDIR = f"{RES}/analysis_v2"
TARGETS = ["Tc (K)", "Pc (bar)", "omega (-)"]
VARIANTS = ["XGB-A", "XGB-B", "XGB-C", "XGB-D"]
CONTRASTS = [("XGB-A", "XGB-B", "estimated supervision"),
             ("XGB-A", "XGB-C", "predicted Tb"),
             ("XGB-C", "XGB-D", "estimated | Tb present"),
             ("XGB-B", "XGB-D", "Tb | estimated present")]
SIM_BINS = [0, 0.4, 0.6, 0.8, 1.01]; SIM_LABELS = ["<0.4", "0.4-0.6", "0.6-0.8", ">0.8"]
RANGE_QUANTILES = [0, 0.25, 0.5, 0.75, 1.0]
NEG_RULE_FACTOR = 0.5          # resolution = factor * median(pooled |error|)
TIE_DELTA = 0.02               # near-neighbor tie flag threshold
# ==========================================
import os, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
RDLogger.DisableLog("rdApp.*")

def short(t): return t.split(" ")[0]

def metric_block(y, p):
    e = np.abs(y - p)
    ss_res = np.sum((y - p) ** 2); ss_tot = np.sum((y - y.mean()) ** 2)
    return dict(n=int(len(y)), R2=float(1 - ss_res/ss_tot) if ss_tot > 0 else np.nan,
                RMSE=float(np.sqrt(np.mean((y-p)**2))), MAE=float(e.mean()),
                MedAE=float(np.median(e)), MaxAE=float(e.max()))

def main():
    os.makedirs(OUTDIR, exist_ok=True)
    for sub in ["parity", "residuals", "error_dist", "range", "doa", "top20"]:
        os.makedirs(f"{OUTDIR}/{sub}", exist_ok=True)

    bench = pd.read_csv(f"{ART}/benchmark_split_v3.csv")
    mets_saved = pd.read_csv(f"{RES}/variant_test_metrics.csv")
    preds = pd.read_csv(f"{RES}/variant_test_predictions.csv")
    preds["residual"] = preds.actual - preds.pred          # actual - predicted
    preds["abs_error"] = preds.residual.abs()

    # ---- similarity of each test molecule to experimental TRAIN molecules ----
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fp = {cs: gen.GetFingerprint(Chem.MolFromSmiles(cs)) for cs in bench.canonical_smiles}
    train_cs = bench.loc[bench.split == "train", "canonical_smiles"].tolist()
    train_fps = [fp[cs] for cs in train_cs]
    sim_info = {}
    for cs in preds.canonical_smiles.unique():
        sims = np.array(DataStructs.BulkTanimotoSimilarity(fp[cs], train_fps))
        order = np.argsort(sims)[::-1]
        top1, top2 = sims[order[0]], (sims[order[1]] if len(sims) > 1 else np.nan)
        sim_info[cs] = (float(top1), train_cs[order[0]], float(top2),
                        bool(abs(top1 - top2) <= TIE_DELTA) if len(sims) > 1 else False)
    preds["max_tanimoto"] = preds.canonical_smiles.map(lambda c: sim_info[c][0])
    preds["nearest_train_smiles"] = preds.canonical_smiles.map(lambda c: sim_info[c][1])
    preds["second_max_tanimoto"] = preds.canonical_smiles.map(lambda c: sim_info[c][2])
    preds["near_neighbor_tie"] = preds.canonical_smiles.map(lambda c: sim_info[c][3])

    # ---- (1) full metrics table (recomputed from predictions; best_iteration merged as diagnostic) ----
    rows = []
    for v in VARIANTS:
        for t in TARGETS:
            d = preds[(preds.variant == v) & (preds.target == t)]
            rows.append({"variant": v, "target": t, **metric_block(d.actual.values, d.pred.values)})
    metrics = pd.DataFrame(rows)
    if "best_iteration" in mets_saved.columns:
        metrics = metrics.merge(mets_saved[["variant", "target", "best_iteration"]],
                                on=["variant", "target"], how="left")
        metrics = metrics.rename(columns={"best_iteration": "best_iteration_DIAGNOSTIC_ONLY"})
    metrics.round(4).to_csv(f"{OUTDIR}/01_metrics_all.csv", index=False)

    # ---- (2) ablation-effect table + negligibility ----
    abl = []
    for base, var, desc in CONTRASTS:
        for t in TARGETS:
            db = preds[(preds.variant == base) & (preds.target == t)]
            dv = preds[(preds.variant == var) & (preds.target == t)]
            mb, mv = metric_block(db.actual.values, db.pred.values), metric_block(dv.actual.values, dv.pred.values)
            pooled = np.concatenate([db.abs_error.values, dv.abs_error.values])
            resolution = NEG_RULE_FACTOR * float(np.median(pooled))
            dR = mb["RMSE"] - mv["RMSE"]; dM = mb["MAE"] - mv["MAE"]   # positive => var improves
            neg = (abs(dR) < resolution) and (abs(dM) < resolution)
            abl.append({"contrast": f"{base}->{var}", "isolates": desc, "target": t,
                        "RMSE_base": round(mb["RMSE"], 4), "RMSE_var": round(mv["RMSE"], 4),
                        "dRMSE_abs": round(dR, 4), "dRMSE_pct": round(100*dR/mb["RMSE"], 2),
                        "MAE_base": round(mb["MAE"], 4), "MAE_var": round(mv["MAE"], 4),
                        "dMAE_abs": round(dM, 4), "dMAE_pct": round(100*dM/mb["MAE"], 2),
                        "pooled_medAE": round(float(np.median(pooled)), 4),
                        "resolution(0.5*medAE)": round(resolution, 4),
                        "classification": "NEGLIGIBLE" if neg else "NON-NEGLIGIBLE"})
    abl_df = pd.DataFrame(abl); abl_df.to_csv(f"{OUTDIR}/02_ablation_effects.csv", index=False)

    # ---- (3) parity plots (no Tanimoto colour) ----
    for v in VARIANTS:
        fig, axes = plt.subplots(1, len(TARGETS), figsize=(5*len(TARGETS), 4.5))
        for ax, t in zip(np.atleast_1d(axes), TARGETS):
            d = preds[(preds.variant == v) & (preds.target == t)]
            m = metric_block(d.actual.values, d.pred.values)
            ax.scatter(d.actual, d.pred, s=12, alpha=0.6, color="steelblue")
            lo, hi = min(d.actual.min(), d.pred.min()), max(d.actual.max(), d.pred.max())
            ax.plot([lo, hi], [lo, hi], "k-", lw=1)
            ax.set_xlabel("Actual"); ax.set_ylabel("Predicted"); ax.set_title(f"{v}  {t}")
            ax.text(0.05, 0.95, f"n={m['n']}\nR2={m['R2']:.3f}\nRMSE={m['RMSE']:.3f}\nMAE={m['MAE']:.3f}",
                    transform=ax.transAxes, va="top", fontsize=9,
                    bbox=dict(boxstyle="round", fc="white", alpha=0.8))
        plt.tight_layout(); plt.savefig(f"{OUTDIR}/parity/parity_{v}.png", dpi=150); plt.close()

    # ---- (4) residual-vs-predicted + bias table ----
    bias = []
    for v in VARIANTS:
        fig, axes = plt.subplots(1, len(TARGETS), figsize=(5*len(TARGETS), 4))
        for ax, t in zip(np.atleast_1d(axes), TARGETS):
            d = preds[(preds.variant == v) & (preds.target == t)]
            ax.scatter(d.pred, d.residual, s=12, alpha=0.6, color="darkorange")
            ax.axhline(0, color="k", lw=1)
            ax.set_xlabel("Predicted"); ax.set_ylabel("Residual (actual - pred)"); ax.set_title(f"{v}  {t}")
            bias.append({"variant": v, "target": t,
                         "mean_residual": round(float(d.residual.mean()), 4),
                         "median_residual": round(float(d.residual.median()), 4)})
        plt.tight_layout(); plt.savefig(f"{OUTDIR}/residuals/residuals_{v}.png", dpi=150); plt.close()
    pd.DataFrame(bias).to_csv(f"{OUTDIR}/03_residual_bias.csv", index=False)

    # ---- (5) absolute-error tail: quantiles + ECDF ----
    tail = []
    for v in VARIANTS:
        fig, axes = plt.subplots(1, len(TARGETS), figsize=(5*len(TARGETS), 4))
        for ax, t in zip(np.atleast_1d(axes), TARGETS):
            d = preds[(preds.variant == v) & (preds.target == t)]
            e = np.sort(d.abs_error.values)
            ax.plot(e, np.arange(1, len(e)+1)/len(e), color="purple")
            ax.set_xlabel("|error|"); ax.set_ylabel("ECDF"); ax.set_title(f"{v}  {t}")
            q = np.percentile(d.abs_error.values, [50, 75, 90, 95, 99])
            tail.append({"variant": v, "target": t, "P50": round(q[0], 4), "P75": round(q[1], 4),
                         "P90": round(q[2], 4), "P95": round(q[3], 4), "P99": round(q[4], 4),
                         "MaxAE": round(float(d.abs_error.max()), 4)})
        plt.tight_layout(); plt.savefig(f"{OUTDIR}/error_dist/ecdf_{v}.png", dpi=150); plt.close()
    pd.DataFrame(tail).to_csv(f"{OUTDIR}/04_abs_error_quantiles.csv", index=False)

    # ---- (6) target-range analysis (quantile bins of ACTUAL value) ----
    rng = []
    for v in VARIANTS:
        for t in TARGETS:
            d = preds[(preds.variant == v) & (preds.target == t)].copy()
            edges = d["actual"].quantile(RANGE_QUANTILES).to_numpy(copy=True)
            edges[0] -= 1e-9
            d["range_bin"] = pd.cut(d.actual, np.unique(edges), include_lowest=True)
            for b, g in d.groupby("range_bin"):
                if len(g) == 0: continue
                rng.append({"variant": v, "target": t, "actual_bin": str(b), "n": len(g),
                            "MAE": round(float(g.abs_error.mean()), 4),
                            "RMSE": round(float(np.sqrt((g.residual**2).mean())), 4)})
    pd.DataFrame(rng).to_csv(f"{OUTDIR}/05_error_by_target_range.csv", index=False)

    # ---- (7) domain of applicability ----
    doa = []
    for v in VARIANTS:
        fig, axes = plt.subplots(1, len(TARGETS), figsize=(5*len(TARGETS), 4))
        for ax, t in zip(np.atleast_1d(axes), TARGETS):
            d = preds[(preds.variant == v) & (preds.target == t)]
            ax.scatter(d.max_tanimoto, d.abs_error, s=12, alpha=0.6, color="teal")
            ax.set_xlabel("max Tanimoto to train"); ax.set_ylabel("|error|"); ax.set_title(f"{v}  {t}")
            sp = spearmanr(d.max_tanimoto, d.abs_error); pe = pearsonr(d.max_tanimoto, d.abs_error)
            dd = d.copy(); dd["sim_bin"] = pd.cut(dd.max_tanimoto, SIM_BINS, labels=SIM_LABELS)
            for b, g in dd.groupby("sim_bin"):
                doa.append({"variant": v, "target": t, "sim_bin": str(b), "n": len(g),
                            "MAE": round(float(g.abs_error.mean()), 4) if len(g) else np.nan,
                            "RMSE": round(float(np.sqrt((g.residual**2).mean())), 4) if len(g) else np.nan,
                            "spearman_all": round(float(sp.correlation), 3),
                            "pearson_all": round(float(pe[0]), 3),
                            "small_n_flag": len(g) < 10})
        plt.tight_layout(); plt.savefig(f"{OUTDIR}/doa/error_vs_similarity_{v}.png", dpi=150); plt.close()
    pd.DataFrame(doa).to_csv(f"{OUTDIR}/06_domain_of_applicability.csv", index=False)

    # ---- (8) top-20 abs-error per variant-target ----
    for v in VARIANTS:
        for t in TARGETS:
            d = preds[(preds.variant == v) & (preds.target == t)]
            cols = ["canonical_smiles", "cluster_id", "actual", "pred", "residual", "abs_error",
                    "max_tanimoto", "second_max_tanimoto", "near_neighbor_tie", "nearest_train_smiles"]
            cols = [c for c in cols if c in d.columns]
            d.nlargest(20, "abs_error")[cols].to_csv(
                f"{OUTDIR}/top20/top20_{v}_{short(t)}.csv", index=False)

    # ---- target-specific model selection (RMSE primary, negligibility-aware) ----
    selection = {}
    for t in TARGETS:
        mt = metrics[metrics.target == t].sort_values("RMSE").reset_index(drop=True)
        best = mt.iloc[0]
        pooled_all = preds[preds.target == t].abs_error.values
        resolution = NEG_RULE_FACTOR * float(np.median(pooled_all))
        similar = mt[(mt.RMSE - best.RMSE).abs() < resolution]["variant"].tolist()
        selection[t] = {"ranked_by_RMSE": mt[["variant", "RMSE", "MAE", "MedAE", "MaxAE"]].round(4).to_dict("records"),
                        "lowest_RMSE_variant": best.variant,
                        "resolution(0.5*medAE)": round(resolution, 4),
                        "effectively_similar_to_best": similar,
                        "verdict": (f"{best.variant} lowest RMSE; {similar} effectively similar on this split"
                                    if len(similar) > 1 else f"{best.variant} distinct best on this split")}
    json.dump(selection, open(f"{OUTDIR}/07_model_selection.json", "w"), indent=2)

    # ---- machine-generated Markdown summary ----
    lines = ["# XGB A/B/C/D Ablation — Analysis Summary (Frozen Protocol v3)", "",
             "**Single frozen split.** No error bars or statistical-significance claims. All deltas are "
             "*observed on this frozen test split* only.", "",
             "**Residual sign:** residual = actual - predicted (positive => under-prediction).", "",
             "**Ablation improvement convention:** delta = metric(baseline) - metric(variant); "
             "positive => variant reduces error.", "",
             "**Negligibility rule:** for each contrast+target, resolution = 0.5 x median(pooled |error| of the two "
             "variants). |dRMSE| and |dMAE| both below resolution => NEGLIGIBLE on this split.", "",
             "## Ablation effects"]
    for _, r in abl_df.iterrows():
        lines.append(f"- **{r['contrast']}** ({r['isolates']}), {r['target']}: "
                     f"dRMSE={r['dRMSE_abs']:+} ({r['dRMSE_pct']:+}%), dMAE={r['dMAE_abs']:+} "
                     f"({r['dMAE_pct']:+}%); resolution={r['resolution(0.5*medAE)']} -> **{r['classification']}**")
    lines += ["", "## Target-specific selection (RMSE primary, negligibility-aware)"]
    for t in TARGETS:
        s = selection[t]
        lines.append(f"- **{t}**: {s['verdict']} (resolution={s['resolution(0.5*medAE)']}).")
    lines += ["", "_Interpretation restricted to what a single frozen split supports. "
              "'Effectively similar' variants should not be ranked as winners._"]
    open(f"{OUTDIR}/00_SUMMARY.md", "w").write("\n".join(lines))

    print("Metrics:\n", metrics.round(4).to_string(index=False))
    print("\nAblation:\n", abl_df[["contrast", "target", "dRMSE_abs", "dRMSE_pct", "classification"]].to_string(index=False))
    print("\nSelection verdicts:")
    for t in TARGETS: print(f"  {t}: {selection[t]['verdict']}")
    print(f"\nAll analysis written to {OUTDIR}/  (old outputs preserved)")

if __name__ == "__main__":
    main()

Metrics:
 variant    target   n     R2    RMSE     MAE   MedAE    MaxAE  best_iteration_DIAGNOSTIC_ONLY
  XGB-A    Tc (K) 209 0.9235 31.1722 19.8383 11.0042 133.0551                             789
  XGB-A  Pc (bar) 195 0.8816  3.5689  1.8778  0.9024  21.8438                             467
  XGB-A omega (-) 157 0.9197  0.0829  0.0495  0.0199   0.3590                             337
  XGB-B    Tc (K) 209 0.9383 27.9880 18.4898 11.3592 144.9265                             190
  XGB-B  Pc (bar) 195 0.8754  3.6614  1.8777  0.8601  26.8268                             102
  XGB-B omega (-) 157 0.9137  0.0860  0.0460  0.0142   0.3736                             498
  XGB-C    Tc (K) 209 0.9457 26.2513 17.0181 10.9718 123.7222                            1052
  XGB-C  Pc (bar) 195 0.8782  3.6197  1.8901  0.8806  23.8456                             515
  XGB-C omega (-) 157 0.9226  0.0814  0.0485  0.0200   0.3471                             578
  XGB-D    Tc (K) 209 0.9505 25.0769 16.2905 10.75